In [2]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")


Current GPU usage:
 - GPU0: 0B



In [3]:
def scheduler(epoch, lr):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-1e-4))

In [4]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [5]:
unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.01/3090, 'dnuSer':0.01/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [6]:
loss_func = ['WMSE']
n_layers = [6]
units = [128]
Nepochs = 1000
learning_rate = [0.0001, 0.00001, 0.00005, 0.0005]
arch_df = pd.DataFrame(product(n_layers,units,loss_func, learning_rate, ))

In [7]:
arch_df

,0,1,2,3
0,6,128,WMSE,0.00010
1,6,128,WMSE,0.00001
2,6,128,WMSE,0.00005
3,6,128,WMSE,0.00050


In [8]:
for i in arch_df.index:
    ######## define architecture:
    model_name = 'WMSE-numax-dnu'
    n_dense_layers = arch_df.loc[i, 0] #number of dense layers
    dense_layer_units = arch_df.loc[i, 1] #neurons per dense layer
    loss_func = arch_df.loc[i, 2]
    learning_rate = arch_df.loc[i, 3]

    ###### Checkpointing

    checkpoint_dir = f'./checkpoints/short-run-tests/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}.model.keras'

    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1)

    lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

    ######## map out model architecture
    #### input layer
    nn_input = keras.Input(shape=(len(inputs),))

    #### dense layer(s)
    for n_dense_layer in range(n_dense_layers):
        if n_dense_layer == 0:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
        else:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

    #### output layer
    nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

    ######## store architecture as keras model
    model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

    tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'../logs/short-run-tests/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}')

    model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

    history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**16, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback]) 

Epoch 1/1000


I0000 00:00:1743690376.605321 3801921 service.cc:148] XLA service 0x7e2f2c002760 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743690376.605345 3801921 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-03 15:26:16.642071: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1743690376.777874 3801921 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-03 15:26:16.822065: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-03 15:26:17.44629

 28/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5982942.0000

I0000 00:00:1743690378.936617 3801921 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 94/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5624976.5000

2025-04-03 15:26:20.392818: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_377', 24 bytes spill stores, 48 bytes spill loads

2025-04-03 15:26:20.487311: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_453', 24 bytes spill stores, 24 bytes spill loads

2025-04-03 15:26:20.518279: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-04-03 15:26:20.576899: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_453_0', 36 bytes spill stores, 36 bytes spill loads

2025-04-03 15:26:20.581424: I external/local_xla/xla/stream_ex

104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 5602301.5000

2025-04-03 15:26:22.716705: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-04-03 15:26:22.871214: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 616 bytes spill stores, 440 bytes spill loads




Epoch 1: saving model to ./checkpoints/short-run-tests/chk-WMSE-numax-dnu-nlayers-6-nunits-128-epochs-1000-lrate-0.0001.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - loss: 5600210.0000 - val_loss: 5264098.5000
Epoch 2/1000
 97/104 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5262429.5000
Epoch 2: saving model to ./checkpoints/short-run-tests/chk-WMSE-numax-dnu-nlayers-6-nunits-128-epochs-1000-lrate-0.0001.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5260554.5000 - val_loss: 5199414.0000
Epoch 3/1000
 95/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5180815.0000
Epoch 3: saving model to ./checkpoints/short-run-tests/chk-WMSE-numax-dnu-nlayers-6-nunits-128-epochs-1000-lrate-0.0001.model.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 5180266.5000 - val_loss: 5151456.0000
Epoch 4/1000
 95/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5144163.0000
Epoch 4: saving model to ./checkpoints/short-run-tests/chk-WMSE-numax-dnu-nlayers-6-nunits-128-epochs-1000-lrate-0.0

KeyboardInterrupt: 